In [5]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
import netCDF4 as nc
import os


In [ ]:
# train: 2000–2018
# val: 2019–2021
# test: 2022–2025

In [2]:
y_path = "../.data/GRIDMET/gridmet_4km_jja_ne/tmmx_JJA_NE_2000_2025.nc"
ds_grid = xr.open_dataset(y_path)


In [3]:
x_path = "../.data/ERA5/era5_air_temperature_JJA_NE_2000_2025_masked.nc"
ds_era5_sub_masked = xr.open_dataset(x_path)

In [4]:
topo_path = "../.data/ETOPO2/topography_on_gridmet_masked.nc"
ds_topo = xr.open_dataset(topo_path)

In [6]:


# =========================================================
# helpers
# =========================================================
def standardize_time_dim(ds):
    """Rename any day-like time dimension/coord to 'time'."""
    rename_dict = {}

    if "day" in ds.dims:
        rename_dict["day"] = "time"
    if "day" in ds.coords:
        rename_dict["day"] = "time"

    if "datetime" in ds.dims:
        rename_dict["datetime"] = "time"
    if "datetime" in ds.coords:
        rename_dict["datetime"] = "time"

    if rename_dict:
        ds = ds.rename(rename_dict)

    return ds


def get_main_var_name(ds):
    """Return the only data variable, or raise an error if there are many."""
    var_names = list(ds.data_vars)
    if len(var_names) != 1:
        raise ValueError(
            f"Expected exactly 1 data variable, but found {var_names}. "
            "Please choose the correct one manually."
        )
    return var_names[0]


def rename_coarse_grid(da):
    """Rename coarse lat/lon dims so they can coexist with high-res lat/lon."""
    rename_dict = {}

    if "lat" in da.dims:
        rename_dict["lat"] = "lat_coarse"
    if "lon" in da.dims:
        rename_dict["lon"] = "lon_coarse"

    if rename_dict:
        da = da.rename(rename_dict)

    return da


def subset_years(da, start_year, end_year):
    """Select a year range inclusive."""
    years = da["time"].dt.year
    return da.sel(time=(years >= start_year) & (years <= end_year))




In [7]:
# =========================================================
# 1. standardize datasets
# =========================================================
ds_grid = standardize_time_dim(ds_grid)
ds_era5_sub_masked = standardize_time_dim(ds_era5_sub_masked)

# detect variable names
y_name = get_main_var_name(ds_grid)
x_name = get_main_var_name(ds_era5_sub_masked)
z_name = get_main_var_name(ds_topo)

print("High-res target variable:", y_name)
print("Low-res input variable:", x_name)
print("Topography variable:", z_name)

# extract DataArrays
y = ds_grid[y_name].rename("tmax_highres")
x = ds_era5_sub_masked[x_name].rename("tmax_lowres")
topo = ds_topo[z_name].rename("elevation")

# sort time just in case
y = y.sortby("time")
x = x.sortby("time")

High-res target variable: air_temperature
Low-res input variable: air_temperature
Topography variable: z


In [8]:
# =========================================================
# 2. align time between low-res and high-res
# =========================================================
common_time = np.intersect1d(
    pd.to_datetime(y["time"].values),
    pd.to_datetime(x["time"].values)
)

print(f"Number of common days: {len(common_time)}")

y = y.sel(time=common_time)
x = x.sel(time=common_time)

Number of common days: 2392


In [9]:
# =========================================================
# 3. rename coarse grid dims so low-res and high-res
#    can live in the same Dataset
# =========================================================
x = rename_coarse_grid(x)

In [10]:
# =========================================================
# 4. build a fixed valid mask on the high-res grid
#    (1 = valid US land pixel, 0 = invalid)
# =========================================================
# this assumes topo is masked outside the valid domain
valid_mask = (np.isfinite(topo) & y.notnull().any(dim="time")).astype("int8")
valid_mask = valid_mask.rename("valid_mask")

print("High-res shape:", y.shape)
print("Low-res shape:", x.shape)
print("Topo shape:", topo.shape)
print("Valid pixels:", int(valid_mask.sum().values))

High-res shape: (2392, 240, 311)
Low-res shape: (2392, 39, 51)
Topo shape: (240, 311)
Valid pixels: 35580


In [15]:
# =========================================================
# 5. define splits
# =========================================================
splits = {
    "train": (2000, 2018),
    "val":   (2019, 2021),
    "test":  (2022, 2025),
}

out_dir = "../.data/downscaling_splits"
os.makedirs(out_dir, exist_ok=True)

In [16]:



# =========================================================
# 6. save each split as one .nc file
# =========================================================
for split_name, (start_year, end_year) in splits.items():
    x_split = subset_years(x, start_year, end_year)
    y_split = subset_years(y, start_year, end_year)

    # safety check
    if x_split.sizes["time"] != y_split.sizes["time"]:
        raise ValueError(
            f"{split_name}: time length mismatch: "
            f"x={x_split.sizes['time']}, y={y_split.sizes['time']}"
        )

    ds_out = xr.Dataset(
        data_vars={
            "tmax_lowres": x_split.astype("float32"),
            "tmax_highres": y_split.astype("float32"),
            "elevation": topo.astype("float32"),
            "valid_mask": valid_mask.astype("int8"),
        },
        attrs={
            "description": f"{split_name} split for Tmax downscaling",
            "train_period": "2000-2018",
            "val_period": "2019-2021",
            "test_period": "2022-2025",
            "note": "Low-res grid kept on native ERA5 grid using lat_coarse/lon_coarse. "
                    "High-res target and elevation are on the 4 km grid."
        }
    )

    # compression
    encoding = {
        "tmax_lowres": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "tmax_highres": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "elevation": {"zlib": True, "complevel": 4, "dtype": "float32"},
        "valid_mask": {"zlib": True, "complevel": 4, "dtype": "int8"},
    }

    out_path = os.path.join(out_dir, f"{split_name}.nc")
    ds_out.to_netcdf(out_path, engine="netcdf4", encoding=encoding)

    t0 = pd.to_datetime(ds_out.time.values[0]).strftime("%Y-%m-%d")
    t1 = pd.to_datetime(ds_out.time.values[-1]).strftime("%Y-%m-%d")
    print(f"Saved {split_name}: {out_path}")
    print(f"  time range: {t0} to {t1}")
    print(f"  n_time = {ds_out.sizes['time']}")

Saved train: ../.data/downscaling_splits/train.nc
  time range: 2000-06-01 to 2018-08-31
  n_time = 1748
Saved val: ../.data/downscaling_splits/val.nc
  time range: 2019-06-01 to 2021-08-31
  n_time = 276
Saved test: ../.data/downscaling_splits/test.nc
  time range: 2022-06-01 to 2025-08-31
  n_time = 368


In [17]:
ds_train = xr.open_dataset("../.data/downscaling_splits/train.nc")
ds_train

<xarray.Dataset> Size: 536MB
Dimensions:       (time: 1748, lat_coarse: 39, lon_coarse: 51, lat: 240,
                   lon: 311)
Coordinates:
  * time          (time) datetime64[ns] 14kB 2000-06-01 ... 2018-08-31
  * lat_coarse    (lat_coarse) float64 312B 46.75 46.5 46.25 ... 37.5 37.25
  * lon_coarse    (lon_coarse) float64 408B -79.75 -79.5 -79.25 ... -67.5 -67.25
  * lat           (lat) float64 2kB 46.98 46.94 46.9 46.86 ... 37.11 37.07 37.03
  * lon           (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.14 -67.1 -67.06
Data variables:
    tmax_lowres   (time, lat_coarse, lon_coarse) float32 14MB ...
    tmax_highres  (time, lat, lon) float32 522MB ...
    elevation     (lat, lon) float32 299kB ...
    valid_mask    (lat, lon) int8 75kB ...
Attributes:
    description:   train split for Tmax downscaling
    train_period:  2000-2018
    val_period:    2019-2021
    test_period:   2022-2025
    note:          Low-res grid kept on native ERA5 grid using lat_coarse/lon...

In [18]:
ds_train.valid_mask
#valid_mask = 1 means:
# elevation is finite there, and
# the high-res Tmax has at least one valid value over time there

# valid_mask = 0 means:
# outside your valid domain, or
# masked ocean / non-US region, or
# missing everywhere in the high-res target

<xarray.DataArray 'valid_mask' (lat: 240, lon: 311)> Size: 75kB
[74640 values with dtype=int8]
Coordinates:
  * lat      (lat) float64 2kB 46.98 46.94 46.9 46.86 ... 37.11 37.07 37.03
  * lon      (lon) float64 2kB -79.97 -79.93 -79.89 ... -67.14 -67.1 -67.06
Attributes:
    actual_range:       [-10791.   8440.]
    units:              K
    description:        Daily Maximum Temperature
    standard_name:      tmmx
    dimensions:         lon lat time
    grid_mapping:       crs
    coordinate_system:  WGS84,EPSG:4326

In [19]:
mask_any = ds_train["tmax_highres"].notnull().any(dim="time")
mask_all = ds_train["tmax_highres"].notnull().all(dim="time")

print((mask_any != mask_all).sum().item())

0
